
# VHF Radio Agent — From Text to Agentic Training Data
## Auto Pilot Project · VHF Protocol Module

**Pipeline adapted from:**
[IRTM Tutorial 13 — From Text to Agentic Training Data](https://github.com/TextMiningUM/IRTM-Student/tree/main/Assignments/13%20agentic_training_data) (Jan Scholtes, Maastricht University)

---

### What this notebook does, end to end

```
Data/VHF/VHFProtocol/ (45 source docs)      Data/VHF/VHF_Eval/ (held-out, never trained on)
        |                                    540 gold Q&A + 498 COLREG scenarios
        |
        v
§ 1   Held-out eval data loaded (gold answers + expected_points, human-authored, committed)
§ 2   Source docs -> structured JSON               (pipeline.ingest.build_vhf_json)
§ 3   JSON -> RAG chunks + embeddings              (pipeline.ingest.build_rag)
§ 4   Concept Knowledge Graph (KG) over the chunks  (pipeline.ingest.build_kg)
§ 5   Reasoning-trace extraction (1 LLM call/chunk) (pipeline.track1.extract_reasoning)
§ 6   Agentic training data: SFT / multi-hop / DPO / reflection, Track 1 + Track 2
§ 7   Gold-standard atomic-claim enrichment (Claude) -- prerequisite for suite v2
§ 8   Procedural Graph (PG): *what-to-do* knowledge alongside the KG's *what-is*
§ 9   Metrics explained — the claim-level RAGAS suite v2 (one authoritative section)
§ 10  Baseline: QWEN base on BOTH tracks, scored with suite v2
§ 11  Prompt ablation V0-V4 (base / RAG / CoT / RAG+CoT / PG-guidance)
§ 12  Pre-fine-tune checks: data-coverage gaps + cross-source consistency
§ 13  Fine-tuning (QLoRA: SFT -> DPO -> Reflection) -> merge -> eval both tracks
§ 14  Compression: AWQ int4 -> layer pruning -> knowledge distillation
§ 15  Final comparison — every model, both tracks, side by side
```

### Two tracks, from day one

Every stage — training-data generation and evaluation alike — runs **two parallel tracks**, because knowing the rules and actually running a compliant radio conversation are two different, separately measurable skills:

| | Track 1 — Rules & knowledge | Track 2 — Conversational compliance |
|---|---|---|
| Question asked | "Does the model **know** the rule?" | "Can the model **conduct the radio exchange** and stay COLREG-compliant while doing it?" |
| Eval data | `vhf_gold_answers.json` — 540 SRC-exam Q&A + `expected_points` | `vhf_colreg_scenarios.json` — 498 collision-avoidance scenarios, 18 encounter types |
| Training data | `vhf_sft_*` + `vhf_multihop` + `vhf_dpo_pairs` + `vhf_reflection` + `vhf_pg_sft` (§ 6/§ 8) | `vhf_conversations.jsonl` (360 multi-turn dialogues) mined into the same artifact types |
| Eval script | `eval_finetuned.py` | `eval_colreg_scenarios.py` |

Both tracks feed the **same** QLoRA fine-tune (§ 13) — one model, two competencies — but stay in separate files/prompts/eval sets so each is measurable independently, and so Track 2's naturally fluent dialogue anchors against the telegraphic label:value style that once contaminated Track 1's synthetic answers.

**Every model in the pipeline is evaluated on both tracks**, starting from the very first baseline (§ 10) through fine-tuning (§ 13), compression (§ 14) and the final comparison (§ 15) — never just the rules-knowledge number in isolation.

### Two standard checks before every fine-tune (§ 12)

Fine-tuning cannot teach a model something the training data never showed it, and one bad sentence in a source document can get chunked, traced, and multiplied into dozens of wrong training rows. § 12 runs two cheap, repeatable checks against the base-model baseline and the parsed corpus *before* committing GPU time to training:
- **Data-coverage gap analysis** ([analyze_gaps.py](pipeline/eval/analyze_gaps.py)) — ranks sections/categories by mean score, flags whether the worst ones are a missing-content gap, an under-represented demonstration style, or a training issue data won't fix.
- **Cross-source consistency check** ([check_consistency.py](pipeline/eval/check_consistency.py)) — a narrow, high-precision checker for facts with exactly one universally-correct answer (phonetic alphabet, call-repetition counts, digit pronunciation). Found and fixed several self-inflicted generation bugs on this corpus; see § 12 for specifics.

Both are **iterate until it stops paying off**, not one-shot stages — see § 12 for the full workflow.

### Repository layout

```
Auto Pilot/
├── VHF_Agent_Training_Pipeline.ipynb    ← THIS notebook (VHF domain)
├── OOW_Agent_Training_Pipeline.ipynb    ← COLREG/OOW domain, same shape
├── Data/VHF/
│   ├── VHFProtocol/        ← training source documents (auto-discovered)
│   ├── VHF_Eval/           ← held-out eval: gold Q&A, COLREG scenarios, *_claims.json
│   ├── VHF_JSON/           ← one hierarchical JSON per source document (§ 2 output)
│   └── VHF_Agents_Training/← RAG chunks/embeddings, KG, PG, traces, SFT/DPO/reflection,
│                              ablation results, eval summaries (both tracks)
├── _models/
│   ├── hf_cache/           ← Qwen base weights — SHARED across all domains
│   └── VHF/                ← VHF-only fine-tuned artefacts
└── pipeline/               ← all pipeline scripts (ingest / track1 / track2 / eval / train / compress)
```

Training data (`VHFProtocol/`) and eval data (`VHF_Eval/`) live in separate folders so the eval set can't accidentally leak into training; every training-data builder additionally cosine-filters against both held-out eval files (threshold 0.85) as a second line of defence.


---
## § 0 — Environment Setup

Install all required packages. Run once; idempotent thereafter.  
Tier-2 packages (`torch`, `transformers`, `peft`, `trl`) are large — comment them out if you only want to run the Tier-1 prompt-only evaluation.

In [1]:
import subprocess, sys, time
from pathlib import Path

packages = [
    # Core inference (API clients)
    "openai>=1.40.0",
    "anthropic>=0.34.0",
    "tiktoken>=0.7.0",
    # Retrieval / embeddings
    "sentence-transformers>=2.7.0",
    "scikit-learn>=1.4.0",
    # Data
    "pandas>=2.2.0",
    "numpy>=1.26.0",
    "networkx>=3.2",
    # PDF extraction
    "pdfplumber>=0.11.0",
    # Progress + plotting
    "tqdm>=4.66.0",
    "matplotlib>=3.8.0",
    # HuggingFace stack (torch is installed separately in § 0.1 with CUDA support)
    "transformers>=4.40.0",
    "peft>=0.10.0",
    "trl>=0.8.0",
    "accelerate>=0.29.0",
]

t0 = time.time()
print("[setup] Installing / verifying dependencies (excluding torch)...", flush=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print(f"[setup] Done in {time.time()-t0:.1f}s", flush=True)

# bitsandbytes: 4-bit NF4 quantisation for QLoRA. Native Windows support since 0.43+.
# Install separately because it can fail on some platforms and we want a clear error.
print("\n[setup] Installing bitsandbytes (for 4-bit Qwen quantisation)...", flush=True)
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.43.0"])
    print("[setup] bitsandbytes installed ✅")
except subprocess.CalledProcessError as e:
    print(f"[setup] ⚠️  bitsandbytes install failed ({e}). Qwen will load in bf16 instead of 4-bit.")


[setup] Installing / verifying dependencies (excluding torch)...
[setup] Done in 0.9s

[setup] Installing bitsandbytes (for 4-bit Qwen quantisation)...
[setup] bitsandbytes installed ✅


### § 0.1 — Install CUDA-aware PyTorch

Auto-detects NVIDIA GPU via `nvidia-smi` and installs the matching PyTorch wheel:
- **NVIDIA GPU present** → `torch+cu124` (CUDA 12.4 runtime, works with driver ≥ 550)
- **No NVIDIA GPU** → CPU-only build

This is separate from § 0 because `pip install torch>=2.2.0` without an index URL defaults to the CPU wheel — the most common cause of "why is my laptop GPU not being used?"

In [2]:
import subprocess, sys, shutil, re, time

def _detect_nvidia_gpu() -> tuple[bool, str]:
    """Return (has_gpu, info_string) by parsing nvidia-smi."""
    if shutil.which("nvidia-smi") is None:
        return False, "nvidia-smi not found on PATH"
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
             "--format=csv,noheader"],
            text=True, timeout=10,
        ).strip()
        return True, out
    except Exception as e:
        return False, f"nvidia-smi failed: {e}"


def _current_torch_status() -> str:
    """Return a short description of currently installed torch, or 'not installed'."""
    try:
        import torch as _t
        return f"torch=={_t.__version__} (cuda_available={_t.cuda.is_available()})"
    except Exception:
        return "not installed"


has_gpu, gpu_info = _detect_nvidia_gpu()
print(f"[gpu-detect] {gpu_info}")
print(f"[torch-current] {_current_torch_status()}")

# Decide which index URL to use
if has_gpu:
    index_url = "https://download.pytorch.org/whl/cu124"
    print(f"[install] NVIDIA GPU detected — installing torch from {index_url}")
else:
    index_url = "https://download.pytorch.org/whl/cpu"
    print(f"[install] No NVIDIA GPU — installing CPU-only torch from {index_url}")

# Only (re)install if current torch does not match desired CUDA state
try:
    import torch
    need_reinstall = has_gpu != torch.cuda.is_available()
except ImportError:
    need_reinstall = True

if need_reinstall:
    t0 = time.time()
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "torch", "torchvision", "torchaudio",
        "--index-url", index_url,
    ])
    print(f"[install] Done in {time.time()-t0:.1f}s — RESTART THE KERNEL, then re-run.")
else:
    print("[install] Skipped: existing torch already matches the desired CUDA state.")


[gpu-detect] NVIDIA GeForce RTX 4070 Laptop GPU, 596.08, 8188 MiB
[torch-current] torch==2.6.0+cu124 (cuda_available=True)
[install] NVIDIA GPU detected — installing torch from https://download.pytorch.org/whl/cu124
[install] Skipped: existing torch already matches the desired CUDA state.


### § 0.2 — GPU sanity check

Verifies that PyTorch actually sees the GPU and can allocate on it.  
If this cell prints `❌ CUDA NOT AVAILABLE` you'll fall back to CPU inference for Qwen2.5-7B — technically works but takes 20+ hours for 540 questions.

Common fixes if CUDA is missing:
- **Re-run § 0.1 and restart the kernel** — pip may have installed the CPU wheel earlier
- **Update your NVIDIA driver** to ≥ 550 (CUDA 12.4 compatible)
- **Laptop with hybrid graphics** → open Windows Graphics Settings → set Python to *High performance* → restart

In [3]:
import platform, sys, time
import torch

print(f"Python              : {sys.version.split()[0]}")
print(f"Platform            : {platform.platform()}")
print(f"PyTorch             : {torch.__version__}")
print(f"CUDA build          : {torch.version.cuda}")
print(f"cuDNN               : {torch.backends.cudnn.version()}")
print(f"CUDA available      : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    print("\n❌ CUDA NOT AVAILABLE — Qwen2.5-7B will fall back to CPU (very slow).")
    print("   Re-run § 0.1 and restart the kernel, or check NVIDIA driver ≥ 550.")
else:
    n = torch.cuda.device_count()
    print(f"GPU count           : {n}")
    for i in range(n):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Compute capability: {props.major}.{props.minor}")
        print(f"  Total VRAM        : {props.total_memory / 1024**3:.1f} GB")

    # Quick allocate + matmul benchmark
    print("\n[benchmark] fp16 matmul on GPU 0 (2048×2048)...")
    torch.cuda.synchronize()
    t0 = time.time()
    a = torch.randn(2048, 2048, device="cuda:0", dtype=torch.float16)
    b = torch.randn(2048, 2048, device="cuda:0", dtype=torch.float16)
    for _ in range(20):
        c = a @ b
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    tflops = (20 * 2 * 2048**3) / elapsed / 1e12
    print(f"  20× matmul in {elapsed*1000:.1f} ms  →  ~{tflops:.1f} TFLOPS fp16")

    # bitsandbytes sanity check (needed for 4-bit NF4)
    print("\n[bitsandbytes] checking 4-bit quantisation support...")
    try:
        import bitsandbytes as bnb
        print(f"  bitsandbytes {bnb.__version__} ✅")
    except Exception as e:
        print(f"  ⚠️  bitsandbytes not available ({e.__class__.__name__}: {e})")
        print("     Qwen2.5-7B will load in bf16 (~14 GB VRAM) instead of 4-bit (~5 GB).")

    # Rough VRAM budget estimate for Qwen2.5-7B in 4-bit NF4
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"\n[budget] Qwen2.5-7B-Instruct memory footprint:")
    print(f"  4-bit NF4 (recommended) : ~5 GB VRAM  {'✅ fits' if total_gb >= 6 else '❌ tight'}")
    print(f"  bf16                    : ~14 GB VRAM {'✅ fits' if total_gb >= 15 else '❌ too small'}")
    print(f"  fp32                    : ~28 GB VRAM {'✅ fits' if total_gb >= 30 else '❌ too small'}")


Python              : 3.13.14
Platform            : Windows-11-10.0.26200-SP0
PyTorch             : 2.6.0+cu124
CUDA build          : 12.4
cuDNN               : 90100
CUDA available      : True
GPU count           : 1

GPU 0: NVIDIA GeForce RTX 4070 Laptop GPU
  Compute capability: 8.9
  Total VRAM        : 8.0 GB

[benchmark] fp16 matmul on GPU 0 (2048×2048)...
  20× matmul in 133.5 ms  →  ~2.6 TFLOPS fp16

[bitsandbytes] checking 4-bit quantisation support...
  bitsandbytes 0.50.2 ✅

[budget] Qwen2.5-7B-Instruct memory footprint:
  4-bit NF4 (recommended) : ~5 GB VRAM  ✅ fits
  bf16                    : ~14 GB VRAM ❌ too small
  fp32                    : ~28 GB VRAM ❌ too small


---
## § 1 — Parse VHF Exam Questions

The file contains 540 open-ended questions across 27 SRC sections.  
We parse it into a list of dicts: `{id, section_id, section_title, type, question}`.  
Gold reference answers are **not** in the source file — we generate them in § 2.

In [4]:
import re, json
from pathlib import Path

# ── Data locations ────────────────────────────────────────────────────────
# WORKSPACE     : project root (contains scripts, .venv, _models, .env)
# VHF_ROOT      : Data/VHF — all VHF-domain data lives here
# DATA_DIR      : training documents (RAG / SFT / KG / DPO / reflection)
# EVAL_DIR      : held-out evaluation material — NEVER goes into training
# EVAL_FILE     : the 540-question exam text
#
# WORKSPACE is the notebook's own directory, so this cell runs unchanged
# on Windows, on the cloud instance, or under JupyterHub for any teammate.
WORKSPACE = Path.cwd()
VHF_ROOT  = WORKSPACE / "Data" / "VHF"
DATA_DIR  = VHF_ROOT / "VHFProtocol"
EVAL_DIR  = VHF_ROOT / "VHF_Eval"
EVAL_FILE = EVAL_DIR / "VHF Exam Questions.txt"


def parse_exam_questions(path: Path) -> list[dict]:
    """Parse the SRC exam questions file into structured records."""
    lines = path.read_text(encoding="utf-8").splitlines()

    section_re = re.compile(r"^(\d+\.\d+)\s+(.*?)\s+\((Theory|Practical)\s*-\s*(\d+)\)")
    questions   = []
    current     = {}
    q_counter   = 0

    for line in lines:
        line = line.strip()
        if not line:
            continue

        m = section_re.match(line)
        if m:
            current = {
                "section_id":    m.group(1),
                "section_title": m.group(2).strip(),
                "type":          m.group(3),
                "expected_n":    int(m.group(4)),
            }
            q_counter = 0
            continue

        if line.lower() in {"theoretical part", "practical part", "permanent link"}:
            continue

        if current and line:
            q_counter += 1
            sec = current["section_id"]
            questions.append({
                "id":            f"vhf_{sec}_{q_counter:02d}",
                "section_id":    sec,
                "section_title": current["section_title"],
                "type":          current["type"],
                "question":      line,
                "expected_points": [],
                "gold_answer":   "",
            })

    return questions

eval_questions = parse_exam_questions(EVAL_FILE)

# Summary
import pandas as pd
df = pd.DataFrame(eval_questions)
print(f"Eval file              : {EVAL_FILE}")
print(f"Total questions parsed : {len(eval_questions)}")
print(f"Sections               : {df['section_id'].nunique()}")
print(f"Theory / Practical     : {df['type'].value_counts().to_dict()}")
print()
print(df.groupby(["section_id", "section_title", "type"])
         .size()
         .reset_index(name="n_questions")
         .to_string(index=False))

Eval file              : c:\Users\jcsch\Documents\Python\Auto Pilot\Data\VHF\VHF_Eval\VHF Exam Questions.txt
Total questions parsed : 540
Sections               : 27
Theory / Practical     : {'Theory': 300, 'Practical': 240}

section_id                                                            section_title      type  n_questions
       1.1                                               Marine VHF Legal Framework    Theory           20
      1.10                                            Urgency Voice Calls (PAN PAN)    Theory           20
      1.11                                            Safety Voice Calls (SECURITÉ)    Theory           20
      1.12                                                      Routine Voice Calls    Theory           20
      1.13                                          Digital Selective Calling (DSC)    Theory           20
      1.14                                                 GMDSS and Its Components    Theory           20
      1.15               

---
## § 1.5 — Gold reference answers (held out, human-authored)

Every question needs a `gold_answer` (3-6 sentences) and `expected_points` (2-5 atomic facts). These are **hand-authored and committed** at [`vhf_gold_answers.json`](Data/VHF/VHF_Eval/vhf_gold_answers.json) — an LLM-drafted-then-reviewed gold standard, never regenerated at notebook run time. § 7 later enriches this same file with atomic `gold_claims` for the suite-v2 metrics.


import json

GOLD_FILE = EVAL_DIR / "vhf_gold_answers.json"
gold_records = json.loads(GOLD_FILE.read_text(encoding="utf-8"))
by_id = {r["id"]: r for r in gold_records}
eval_questions_with_gold = [by_id.get(q["id"], q) for q in eval_questions]

n_with_gold = sum(1 for q in eval_questions_with_gold if q.get("gold_answer"))
print(f"Loaded {len(gold_records)} gold records from {GOLD_FILE.name}")
print(f"With gold_answer: {n_with_gold} / {len(eval_questions_with_gold)}")

ex = next((q for q in eval_questions_with_gold if q.get("gold_answer")), None)
if ex:
    print(f"\nExample [{ex['id']}]  section {ex['section_id']} — {ex['section_title']}")
    print(f"  Q : {ex['question']}")
    print(f"  A : {ex['gold_answer'][:200]}...")
    print(f"  EP: {ex['expected_points']}")


---
## § 2 — Source documents → structured JSON

Every training source (7 TXT/MD + 38 PDF = 45 documents) is converted into a **hierarchical JSON schema** before any chunking happens: document → chapter → section, each section tagged with a `type` (`rule` · `definition` · `procedure` · `example` · `warning` · `dialogue` · `reference`), extracted `concepts`/`topics` (VHF terminology, channel numbers, prowords, regulations), and `steps` where applicable.

**Why not chunk raw text directly?** A sliding-window chunker cuts through the middle of a MAYDAY procedure, splits a definition from its term, or mixes an SMCP example with an unrelated rule — every downstream artifact (RAG/KG/SFT/DPO) inherits that structural damage. Preserving structure here means § 3's chunker can respect section/type boundaries instead of guessing them from raw characters.

Code: [build_vhf_json.py](pipeline/ingest/build_vhf_json.py) (file classification, concept vocabulary, TXT/MD + PDF parsers, and the cleanup pass that improves section-type detection and applies any human-reviewed consistency corrections from § 12). Idempotent — only reconverts a file if its JSON is missing; the cleanup pass only reruns if a JSON changed since its last run.


In [ ]:
import subprocess, sys, json
from pathlib import Path

W = Path.cwd()
JSON_OUT_DIR = W / "Data" / "VHF" / "VHF_JSON"

subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.ingest.build_vhf_json"], cwd=W)

index = json.loads((JSON_OUT_DIR / "_index.json").read_text(encoding="utf-8"))["documents"]
print(f"\n{len(index)} documents in {JSON_OUT_DIR}")
print(f"  total sections : {sum(d['sections'] for d in index)}")
print(f"  total words    : {sum(d['words'] for d in index):,}")
by_type = {}
for d in index:
    by_type[d["source_type"]] = by_type.get(d["source_type"], 0) + 1
print(f"  by source_type : {by_type}")


Detected 2 new source file(s) without a JSON — running conversion.
  new: CompleteGuideVHF.txt
  new: Securite_Mayday_Repeat_And_Escalation_Sequences.txt


Documents: 100%|██████████| 46/46 [00:00<00:00, 845.23it/s]


══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  45 source documents in _json/
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
                                               source_file    source_type  chapters  sections  words  parsing_notes
                                      36_VHF_Exchanges.pdf      reference         7        29   1796               
                                     36a-makingcontact.pdf      reference         3        23    702 empty_pages: 1
                                                 5CVHF.txt      reference         1         2   1815               
                                     Basic MAYDAY Call.pdf procedure_card         1         1     12               
                                                BoatUS.txt          guide         2         3    910               
                                      CEPT Regula

### § 2.1 — Sample inspection


In [14]:
def peek(doc_id: str, max_sections: int = 4) -> None:
    """Pretty-print a few sections from one converted document."""
    path = JSON_OUT_DIR / f"{doc_id}.json"
    if not path.exists():
        print(f"Not found: {path}")
        return
    d = json.loads(path.read_text(encoding="utf-8"))
    print("═" * 100)
    print(f"  {d['source_file']}  ·  type={d['source_type']}  ·  publisher={d['publisher']}")
    print(f"  chapters={len(d['chapters'])}  sections={d['metadata']['total_sections']}  words={d['metadata']['total_words']}")
    print(f"  primary_topics: {d['metadata']['primary_topics']}")
    print("═" * 100)
    shown = 0
    for ch in d["chapters"]:
        if shown >= max_sections:
            break
        print(f"\n📖  Chapter: {ch['title']}")
        for s in ch["sections"]:
            if shown >= max_sections:
                break
            shown += 1
            print(f"\n  §  [{s['type']:<10}]  {s['title'][:80]}")
            print(f"     concepts : {s['concepts'][:6]}")
            print(f"     topics   : {s['topics']}")
            if s.get("steps"):
                print(f"     steps    : {len(s['steps'])} extracted")
            print(f"     text     : {s['text'][:200]}...")

# Show three contrasting samples
peek("VHF-procedures-Scheldt-area-EN")   # clean markdown
print()
peek("Basic MAYDAY Call")                # procedure card PDF (JSON keeps original spaces)
print()
peek("VHFPro")                           # SMCP guide TXT

════════════════════════════════════════════════════════════════════════════════════════════════════
  VHF-procedures-Scheldt-area-EN.md  ·  type=protocol  ·  publisher=GNA Scheldt
  chapters=1  sections=56  words=5995
  primary_topics: ['basics', 'channels', 'phonetic', 'prowords', 'regulation']
════════════════════════════════════════════════════════════════════════════════════════════════════

📖  Chapter: VHF PROCEDURES

  §  [reference ]  For safe and efficient shipping traffic in the Scheldt are
     concepts : []
     topics   : []
     text     : **For safe and efficient shipping traffic in the Scheldt area**...

  §  [definition]  FOREWORD
     concepts : ['IMO', 'VHF']
     topics   : ['basics', 'regulation']
     text     : These VHF procedures are a guideline for traffic controllers and traffic participants in the Scheldt area. Their aim is to clarify the use of VHF for safe and smooth shipping traffic. By correctly app...

  §  [procedure ]  2. Purpose of a Vessel Traffic S

---
## § 3 — RAG chunks + embeddings

Turns the ~1,450 semantic sections into 300-500 token chunks that respect the boundaries § 2 established: section-atomic, chapter-bounded, `dialogue`/`definition`/`procedure` sections always stand alone, others may merge with an adjacent section only if they share ≥ 50% of their topics. Each chunk is embedded with `all-MiniLM-L6-v2` (`text_with_context`: chapter title + section headers + body, which measurably improves retrieval for short queries like "Channel 70?").

Code: [build_rag.py](pipeline/ingest/build_rag.py) — writes `vhf_rag_chunks.json` + `vhf_rag_embeddings.npy` + `vhf_rag_chunk_ids.json`, all domain-parameterized via `AgentPaths`.


In [ ]:
import subprocess, sys, json
from pathlib import Path
import numpy as np
import pandas as pd

W = Path.cwd()
CACHE_DIR = W / "Data" / "VHF" / "VHF_Agents_Training"
RAG_CHUNKS_FILE = CACHE_DIR / "vhf_rag_chunks.json"
EMBEDDINGS_FILE = CACHE_DIR / "vhf_rag_embeddings.npy"

subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.ingest.build_rag"], cwd=W)

chunks = json.loads(RAG_CHUNKS_FILE.read_text(encoding="utf-8"))
embs = np.load(EMBEDDINGS_FILE)
df = pd.DataFrame(chunks)
print(f"\n{len(chunks)} chunks, embeddings shape {embs.shape}")
print(f"token_count: min={df['token_count'].min()} median={int(df['token_count'].median())} "
      f"max={df['token_count'].max()}")
print(f"sections per chunk: {df['n_sections'].value_counts().sort_index().to_dict()}")


### § 3.1 — Retrieval smoke test


In [ ]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

_sem_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_ids = json.loads((CACHE_DIR / "vhf_rag_chunk_ids.json").read_text(encoding="utf-8"))
chunk_by_id = {c["chunk_id"]: c for c in chunks}


def retrieve(query: str, k: int = 5) -> list[dict]:
    """Cosine-similarity retrieval over the RAG corpus."""
    q_emb = _sem_model.encode([query], normalize_embeddings=True)[0]
    scores = embs @ q_emb
    top_ix = np.argsort(-scores)[:k]
    return [{**chunk_by_id[chunk_ids[i]], "score": float(scores[i])} for i in top_ix]


for q in ["What is VHF Channel 70 used for?", "How do I send a MAYDAY call?",
          "What is the phonetic word for the letter M?"]:
    print("=" * 100)
    print(f"Query: {q}")
    for i, hit in enumerate(retrieve(q, k=3), 1):
        print(f"  {i}. [{hit['score']:.3f}] {hit['source_file']} -> {hit['chapter_title'][:40]!r}")
        print(f"     text: {hit['text'][:140].replace(chr(10), ' ')}...")


---

# § 4 · Knowledge-Graph RAG

Dense-only retrieval (§ 3) has blind spots. § 4 builds a **concept graph** over the chunks to patch these:

- **Nodes**: chunks + ~75 concepts + topics + documents
- **Edges**: `chunk-MENTIONS-concept`, `concept-COOCCURS-concept` (top-5 by weight), `chunk-HAS_TOPIC-topic`, `chunk-ADJACENT-chunk`
- **Aliases** for query-time concept matching (`dsc alert → DSC/Channel 70`, `bridge to bridge → Channel 13`, ...)
- **Hybrid retrieval**: dense top-N ∪ concept hits ∪ 1-hop cooccurrence expansion → ranked by `dense_score + concept_boost + cooccur_boost`

Code: [build_kg.py](pipeline/ingest/build_kg.py).


In [ ]:
# Build/load the KG. Rebuilds if the on-disk KG is stale vs the current chunk corpus.
import subprocess, sys, json
from pathlib import Path

W = Path.cwd()
CACHE = W / "Data" / "VHF" / "VHF_Agents_Training"
KG_FILE     = CACHE / "vhf_kg.json"
CHUNKS_FILE = CACHE / "vhf_rag_chunks.json"


def _kg_is_stale() -> bool:
    """True if the KG's chunk_meta does not match the current chunks corpus."""
    if not KG_FILE.exists():
        return True
    try:
        current_ids = {c["chunk_id"]
                       for c in json.loads(CHUNKS_FILE.read_text(encoding="utf-8"))}
        kg_ids = set(json.loads(KG_FILE.read_text(encoding="utf-8"))["chunk_meta"].keys())
    except Exception:
        return True
    return current_ids != kg_ids


if _kg_is_stale():
    print("KG missing or stale (chunk-id mismatch with vhf_rag_chunks.json) — rebuilding.")
    KG_FILE.unlink(missing_ok=True)
    subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.ingest.build_kg"], cwd=W)
else:
    print(f"KG up to date: {KG_FILE}")

kg = json.loads(KG_FILE.read_text(encoding="utf-8"))
print("KG stats:", kg["stats"])
print("Aliases:", len(kg["aliases"]))

In [ ]:
# Import the retrieval function so the notebook can use it directly.
from pipeline.ingest.build_kg import kg_retrieve
import numpy as np
from sentence_transformers import SentenceTransformer

CACHE = W / "Data" / "VHF" / "VHF_Agents_Training"
_embs = np.load(CACHE / "vhf_rag_embeddings.npy")
_ids  = json.loads((CACHE / "vhf_rag_chunk_ids.json").read_text(encoding="utf-8"))
_sem_model = SentenceTransformer("all-MiniLM-L6-v2")

def kg_search(q: str, k: int = 5):
    hits, q_cons, expanded = kg_retrieve(q, _sem_model, _embs, _ids, kg, k=k)
    print(f"Q: {q}")
    print(f"   concepts hit: {q_cons}  expanded: {expanded}")
    for r, h in enumerate(hits, 1):
        print(f"  {r}. [{h['score']:.3f}] {h['source_file']}  ->  {h['chapter'][:40]!r}")
    return hits

_ = kg_search("What is VHF Channel 70 used for?", k=3)

---

# § 5 · Reasoning-trace extraction

From each chunk we extract a **structured reasoning trace** using `gpt-4o-mini`. This is the **only LLM call** in the entire training-data pipeline; all downstream datasets (SFT / multi-hop / DPO / reflection / PG) are derived deterministically from these traces.

Schema per trace: `situation`, `trigger`, `procedures[step/action/why]`, `constraints`, `prowords_used`, `channels`, `regulations`, `warnings`, `outcomes`, `key_facts`, `question_seeds[angle/text]`.

Code: [extract_reasoning.py](pipeline/track1/extract_reasoning.py). Resume-safe.


In [ ]:
TRACES_FILE = CACHE / "vhf_reasoning_traces.jsonl"

# extract_reasoning.py is resume-safe: it re-reads the traces file and only calls
# the LLM for chunk_ids that are missing. Running it every time keeps traces
# in sync with the current chunk corpus (e.g. after new source docs were added).
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.track1.extract_reasoning"], cwd=W)

n_ok = n_skip = n_err = 0
with TRACES_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r.get("error"): n_err += 1
        elif r.get("skip"): n_skip += 1
        elif r.get("trace"): n_ok += 1
print(f"Traces — usable: {n_ok}  boilerplate-skipped: {n_skip}  errors: {n_err}")

---

# § 6 · Training-data generation (deterministic, no LLM required)

Four Track 1 datasets built from the same traces:

| Script | Output |
|---|---|
| [build_sft.py](pipeline/track1/build_sft.py) | `vhf_sft_direct/cot/rag.jsonl` |
| [build_multihop.py](pipeline/track1/build_multihop.py) | `vhf_multihop.jsonl` |
| [build_rlhf.py](pipeline/track1/build_rlhf.py) | `vhf_dpo_pairs.jsonl` (6 perturbation types incl. `swap_step_order`) |
| [build_reflection.py](pipeline/track1/build_reflection.py) | `vhf_reflection.jsonl` (Draft/Critique/Refined) |

Every synthesized question is checked against the 540 gold questions (cosine ≥ 0.85 → dropped).

Everything above is **Track 1 — Rules & knowledge**, evaluated in § 13.2. § 6.1 builds **Track 2 — Conversational compliance** (`vhf_conversations.jsonl`, 360 multi-turn dialogues), evaluated in § 13.3. § 6.2 mines those same 360 training conversations (never the eval set) into the same artifact types Track 1 gets. Both feed the **same** SFT run (§ 13) but stay in separate files/prompts/eval sets so each competency is measurable independently.


In [ ]:
for mod in ["pipeline.track1.build_sft", "pipeline.track1.build_multihop", "pipeline.track1.build_rlhf", "pipeline.track1.build_reflection"]:
    print(f"\n>>> {mod}")
    subprocess.check_call([sys.executable, "-X", "utf8", "-m", mod], cwd=W)

In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "manifest.py"], cwd=W)

### § 6.1 — Track 2: conversational-compliance training data

Generated by [build_vhf_conversations.py](pipeline/track2/build_vhf_conversations.py), grounded in the same COLREG rule text as the Track 2 scenarios. 18 encounter types × 2 directions (incoming/outgoing) × 10 = 360 conversations, each 2-3 assistant turns. Kept as its own file, never blended into the Track 1 JSONLs.


In [ ]:
# Track 2 (conversational compliance) training data. --resume skips any
# category/direction that already has enough records on disk, so re-running
# this cell is cheap after the first full generation.
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.track2.build_vhf_conversations", "--resume"], cwd=W)

### § 6.2 — Track 2 (continued): mine agentic training data FROM the 360 training conversations

Mines the 360 **training** conversations only — `vhf_colreg_scenarios.json` (498 records) is the Track 2 **evaluation** set and is never touched here, only used as a held-out contamination check.

1. [extract_conversation_reasoning.py](pipeline/track2/extract_conversation_reasoning.py) — one `gpt-4o-mini` call per conversation, same trace schema as § 5, so the Track 1 builders below run against it unchanged via `--traces-file`.
2. `build_sft.py --traces-file vhf_conversation_traces.jsonl --out-prefix vhf_colreg_sft` → single-turn Q→A mined from the dialogues
3. `build_rlhf.py` → `vhf_colreg_dpo_pairs.jsonl`, `build_reflection.py` → `vhf_colreg_reflection.jsonl`
4. `build_multihop.py --traces-file vhf_reasoning_traces.jsonl vhf_conversation_traces.jsonl` — the only builder taking **two** trace files, so a shared concept can pair a protocol excerpt with a real dialogue → genuine cross-track multi-hop questions

All outputs are wired into `train_sft.py`/`train_dpo.py`/`train_reflection.py`, contamination-filtered against both eval files, zero rows dropped on the last run (expected — none of this is sourced from either held-out file).


In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.track2.extract_conversation_reasoning"], cwd=W)

COLREG_SCEN = str(W / "Data" / "VHF" / "VHF_Eval" / "vhf_colreg_scenarios.json")
CONV_TRACES = str(CACHE / "vhf_conversation_traces.jsonl")
PROTO_TRACES = str(CACHE / "vhf_reasoning_traces.jsonl")

subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.track1.build_sft",
                       "--traces-file", CONV_TRACES, "--out-prefix", "vhf_colreg_sft",
                       "--extra-gold-file", COLREG_SCEN], cwd=W)
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.track1.build_rlhf",
                       "--traces-file", CONV_TRACES,
                       "--out-file", str(CACHE / "vhf_colreg_dpo_pairs.jsonl"),
                       "--extra-gold-file", COLREG_SCEN], cwd=W)
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.track1.build_reflection",
                       "--traces-file", CONV_TRACES,
                       "--out-file", str(CACHE / "vhf_colreg_reflection.jsonl"),
                       "--extra-gold-file", COLREG_SCEN], cwd=W)
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.track1.build_multihop",
                       "--traces-file", PROTO_TRACES, CONV_TRACES,
                       "--out-file", str(CACHE / "vhf_colreg_multihop.jsonl"),
                       "--extra-gold-file", COLREG_SCEN], cwd=W)

# Quick self-check: row counts + how many multihop pairs are genuinely cross-track
# (one side from a protocol document, the other from the conversations).
for name in ["vhf_colreg_sft_direct.jsonl", "vhf_colreg_sft_cot.jsonl", "vhf_colreg_sft_rag.jsonl",
             "vhf_colreg_multihop.jsonl", "vhf_colreg_dpo_pairs.jsonl", "vhf_colreg_reflection.jsonl"]:
    p = CACHE / name
    n = sum(1 for _ in p.open("r", encoding="utf-8")) if p.exists() else 0
    print(f"{name:<32} {n:>5} rows")

cross_track = 0
mh_file = CACHE / "vhf_colreg_multihop.jsonl"
if mh_file.exists():
    with mh_file.open("r", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if "vhf_conversations" in r["source_a"] or "vhf_conversations" in r["source_b"]:
                cross_track += 1
    print(f"\nMulti-hop pairs involving a training conversation: {cross_track}")


---

# § 7 · Gold-standard files + atomic-claim enrichment (suite v2 prerequisite)

Every evaluation in this notebook uses the **claim-level RAGAS suite v2** (§ 9), which needs each gold record decomposed into **atomic claims**.

### How a gold-standard record must look

**Schema A — Track 1** (`Data/VHF/VHF_Eval/vhf_gold_answers.json`, 540 records):

```json
{"id": "vhf_1.1_01",
 "section_id": "1.1", "section_title": "Marine VHF Legal Framework",
 "type": "Theory",
 "question": "…one exam-style question…",
 "gold_answer": "…terse but fluent reference answer…",
 "expected_points": ["…one atomic examinable point…", "…"]}
```

**Schema B — Track 2** (`vhf_colreg_scenarios.json`, 498 records): adds `category`, `colreg_rules`, `region`, `own_vessel`, `target_vessel`, `scenario`, `vhf_channel {hailing, working, settings, note}` — the channel/rule fields are **answer-only**.

Both files are held out: nothing derived from them may ever enter training data.

### The `gold_claims` enrichment (sibling `*_claims.json` files)

[enrich_gold_claims.py](pipeline/eval/enrich_gold_claims.py) asks **Claude** (same provenance as the gold answers, so the GPT judge never grades its own model family's annotations) to decompose each record into 3-10+ atomic claims:

```json
"gold_claims": [
  {"claim": "The initial hail is transmitted on VHF channel 16",
   "type": "number", "value": "16", "role": "vhf_channel"},
  {"claim": "Own vessel alters course to starboard",
   "type": "direction", "value": "starboard", "role": "turn_direction"},
  {"claim": "The transmission ends with the proword OVER",
   "type": "literal", "value": "OVER"},
  {"claim": "The give-way vessel takes early and substantial action",
   "type": "procedure"}
]
```

Rules (enforced by [gold_claims.py](pipeline/eval/gold_claims.py)): atomic & self-contained, grounded only in the record, every `expected_points` entry covered, every safety-critical number a `type:"number"` claim with a role from the fixed vocabulary (`vhf_channel | colreg_rule | distance_nm | speed_kn | bearing_deg | course_deg | power_watt | repeat_count | time_interval | frequency_mhz | gross_tonnage | sea_area | mmsi | other_number`; directions use `turn_direction | pass_side | light_colour`). Internal gold inconsistencies get a `QA_FLAG` claim for human review — the first pass surfaced 59 flags on the scenario file, ~15 of them candidate genuine COLREG errors, queued for review.


In [ ]:
# § 12.9 code: enrich both gold files with atomic claims (Anthropic API, needs
# ANTHROPIC_API_KEY in .env), then validate structurally. Resume-safe: already-
# enriched records are skipped, rejected records are retried on rerun — so
# re-running this cell after adding/fixing gold records only processes the delta.
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.enrich_gold_claims"], cwd=W)
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.gold_claims", "--check",
                       str(W / "Data/VHF/VHF_Eval/vhf_gold_answers_claims.json"),
                       str(W / "Data/VHF/VHF_Eval/vhf_colreg_scenarios_claims.json")], cwd=W)


---

# § 8 · Procedural Graph — *what-to-do* knowledge alongside the KG's *what-is* (Lu et al. 2026)

VHF procedures are inherently **order-critical** (DSC alert *before* voice MAYDAY; hail on 16 *before* switching to a working channel), but the KG (§ 4) can't express order, and nothing before this measured a correctly-worded-but-wrong-order answer. Following the Procedural Graphs paper (Lu et al., Google 2026 — `Data/OOW/OOW_Literature_Review/`), [build_pg.py](pipeline/ingest/build_pg.py) builds a graph of **(procedure, NEXT, procedure)** triplets from § 5's ordered `procedures` lists, mapping the paper's three edge attributes directly onto trace fields already present: `constraints`→`condition`, step-`why`→`guidance`, `warnings`→`pitfalls`.

Deterministic (no LLM): step actions are canonicalized across documents via embedding clustering, with a **safety guard** that never merges steps whose channel numbers or port/starboard directions differ, vessel-name masking, and verb-tense normalization. Each trace is tagged with a procedure **family** (distress/urgency/safety/dsc/colreg_encounter/routine_call). VHF: 670 traces → ~1,760 nodes / ~1,650 edges.

**Three consumers:**
1. **`v4_pg` ablation config** (§ 11): [pg_guidance.py](pipeline/ingest/pg_guidance.py) renders a compact "Procedure guidance" block (typical family sequence + conditions + pitfalls) in the prompt.
2. **Step-order training data**: [build_pg_sft.py](pipeline/track1/build_pg_sft.py) → `vhf_pg_sft.jsonl` (next-step/prerequisite/walkthrough Q&A, contamination-filtered, wired into `train_sft.py`), plus a 6th DPO perturbation axis `swap_step_order`.
3. **`ProcOrder` metric** (§ 9): answer sentences matched to PG nodes; concordant if the graph has a NEXT-path in that direction, a violation if only the reverse path exists. Reported, not yet in the composite (see § 9's caveat).

**Self-evolution** ([evolve_pg.py](pipeline/train/evolve_pg.py)) — the paper's offline refine loop (rollout with guidance → LLM-refiner edits → preserve-or-improve validation gating → rejection memory) is implemented but deferred to the MOOS-in-the-loop phase, where real execution trajectories provide the feedback signal.


In [ ]:
# § 12.10 code: bouw de Procedural Graph en de step-order trainingsdata.
# Beide deterministisch en her-runbaar; build_pg_sft filtert tegen BEIDE eval-files.
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.ingest.build_pg"], cwd=W)
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.track1.build_pg_sft",
                       "--extra-gold-file", str(W / "Data/VHF/VHF_Eval/vhf_colreg_scenarios.json")], cwd=W)


---

# § 9 · Metrics explained — the claim-level RAGAS suite v2

Every evaluation in this notebook (§ 10 baseline, § 11 ablation, § 13 fine-tune eval) is scored with **one shared metric suite** ([ragas_metrics.py](pipeline/eval/ragas_metrics.py)), computed against the atomic `gold_claims` produced in § 7. This section is the single authoritative explanation — later sections only note config-specific composite weights, they don't re-define the metrics.

### Why not the `ragas` pip package

`ragas` 0.4.3 force-downgrades `openai` 3.x→1.x (via an `instructor<2` pin) and pulls in the entire `langchain`/`langgraph` stack, which would break every other script in this pipeline. The metric *algorithms* are published and reimplemented here 1:1, with domain-adapted extraction prompts.

### The metrics

| Metric | Applies to | What it measures | Cost |
|---|---|---|---|
| **AnswerCorrectness** | all configs | answer decomposed into statements, classified TP/FP/FN against `gold_claims` → claim-F1, blended 0.75·F1 + 0.25·SemSim | judge call |
| **ClaimPrec / ClaimRec / ClaimF1** | all configs | the raw P/R/F1 behind AnswerCorrectness, reported separately — Prec = hallucination side (are the model's claims supported?), Rec = completeness side (are all gold claims covered?) | (same call) |
| **Faithfulness** | RAG configs only | answer statements verified against the **retrieved chunks** — real RAGAS faithfulness | judge call |
| **ContextPrecision / ContextRecall** | RAG configs only | were retrieved chunks useful / did retrieval fetch every gold claim? Separates *retrieval failure* from *generation failure* | judge call |
| **CorpusGrounded** | closed-book configs | RAGAS-inspired (not RAGAS faithfulness): each statement verified against its own top-k retrieval from the full corpus — hallucination detection with no context to lean on | judge call |
| **AnswerRelevancy** | all configs | 3 reverse-generated questions vs. the original; 0 if the answer is noncommittal | judge call |
| **NumericF1** (+ NumericPrec / NumericRec) | all configs | deterministic, role-aware (`vhf_channel`, `colreg_rule`, `distance_nm`, …): precision catches **invented** channels/rule numbers, recall catches **missing** ones | free (regex) |
| **LitHit** | all configs | fraction of gold literal claims (prowords) present | free (string match) |
| **Cover** | all configs | sentence-level embedding recall of `expected_points` | free (embedding) |
| **ProcOrder** | all configs, reported only | step-order concordance of answer sentences against the Procedural Graph's NEXT-paths (§ 8) | free (embedding match) — **not yet in the composite**, see caveat below |
| SemSim, AnsRelCos | all configs, reported only | the two free cosine diagnostics from the original Tutorial-13 engine (answer↔gold, answer↔question) — kept visible for continuity, but SemSim is already blended into AnswerCorrectness so isn't double-counted | free (embedding) |

**Composite** is a **fixed weight-set per config kind** (never re-weighted per row — a row with a failed judge call gets `Composite = NaN`, excluded and counted, not silently rebalanced):
- RAG configs: AnswerCorrectness .35 · Faithfulness .20 · ContextRecall .10 · NumericF1 .15 · Cover .10 · AnswerRelevancy .05 · LitHit .05
- Closed-book configs: AnswerCorrectness .35 · CorpusGrounded .20 · NumericF1 .15 · Cover .10 · AnswerRelevancy .05 · LitHit .05
- Track 2 (behavioral): ColregCorrect .30 · AnswerCorrectness .25 · ChannelProc .15 · CallFormatOK .10 · NumericF1 .10 · CorpusGrounded .10

**Caveat — ProcOrder is not yet load-bearing.** Its mean currently sits near 1.0 across evaluations because most answers match fewer than 2 sentences onto Procedural-Graph nodes and fall back to the vacuous "no ordering claim to check" value of 1.0. It needs a non-vacuous-rate check and a calibrated match threshold before it can safely enter the composite — until then, treat it as a diagnostic, not a score.

### Judge caching, suite stamping, and `--legacy`

Every judge verdict is cached on disk (`ragas_judge_cache.jsonl`) keyed by `sha1(kind|model|prompt)`, so re-scoring an existing answer set — or scoring it on a different machine that shares the cache — costs no new API calls. Every summary JSON carries `metric_suite` (`ragas_v2.0`) + `judge_model`; never compare a `ragas_v2.0` number against the historical `legacy_v1` one (the old suite's binary whole-answer `Faith` judge floored around 0.35 even for the base model — it mostly measured "does the answer restate gold verbatim", not hallucination). `--legacy` on any eval script still runs the old suite for exactly that historical comparison.

Paired-bootstrap 95% CIs are reported for every config/checkpoint comparison, so a delta within the CI is flagged as noise rather than presented as a real improvement.


---

# § 10 · Baseline — QWEN base on both tracks, scored with suite v2

Before touching prompts (§ 11) or weights (§ 13), establish what the untuned base model scores under the metrics defined in § 9. Both tracks, no prompt tricks (no RAG, no CoT) — this is the number every later stage must beat.


In [ ]:
# Track 1 -- 540 gold Q&A, closed-book. Skips if already run.
qwen_base_summary = CACHE / "eval_qwen_base_summary.json"
if qwen_base_summary.exists():
    print(f"Skipping -- {qwen_base_summary.name} already exists.")
else:
    subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.eval_finetuned",
                           "--model", "Qwen/Qwen2.5-7B-Instruct", "--tag", "qwen_base"], cwd=W)


In [ ]:
# Track 2 -- 498 COLREG scenarios. Uses --tag qwen_base so § 15's final loop
# detects this file and skips re-running it.
qwen_base_colreg_summary = CACHE / "eval_qwen_base_colreg_summary.json"
if qwen_base_colreg_summary.exists():
    print(f"Skipping -- {qwen_base_colreg_summary.name} already exists.")
else:
    subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.eval_colreg_scenarios",
                           "--model", "Qwen/Qwen2.5-7B-Instruct", "--tag", "qwen_base"], cwd=W)


---

# § 11 · Prompt ablation (before fine-tuning)

Measures how much each knowledge-injection strategy contributes, on the same stratified sample of gold questions, using the metrics defined in § 9:

| Config | System prompt | Context injected? |
|---|---|---|
| **V0** base | plain expert system | no |
| **V1** RAG | "use only these excerpts" | top-3 KG hits |
| **V2** CoT | "think step by step" | no |
| **V3** RAG+CoT | combined | top-3 KG hits |
| **V4** PG | plain expert system | "Procedure guidance" block rendered from the Procedural Graph (§ 8) — questions without a renderable path fall back to the V0 prompt |

VRAM-safe split: prep (embedder only) → run (Qwen 4-bit only) → score (embedder + judge). Scripts: [prep_ablation.py](pipeline/eval/prep_ablation.py), [run_ablation.py](pipeline/eval/run_ablation.py), [score_ablation.py](pipeline/eval/score_ablation.py).

**First full result (n=40 pilot)**: CoT alone (V2, Composite 0.521) matched RAG+CoT (V3, 0.518), both clearly above plain RAG (V1, 0.480) and base (V0, 0.467). The diagnostic reason: **ContextRecall for the RAG configs is only 0.330** — retrieval is fetching barely a third of the gold claims, so the bottleneck at this KG maturity is retrieval quality, not generation. Re-run at full n once the KG/PG are further tuned to see whether this holds.


In [ ]:
# Pilot: 40 questions x 4 configs. For the full 540, pass --n 540.
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.prep_ablation", "--n", "40"], cwd=W)

In [ ]:
# Qwen inference — loads 4-bit NF4, no embedder in parallel (VRAM safe).
# 40 Q x 4 configs x ~7s = ~20 minutes.
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.run_ablation"], cwd=W)

In [ ]:
# Metrics + comparison table. Faith/Correct scored by gpt-4o-mini judge.
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.score_ablation"], cwd=W)

---

# § 12 · Pre-fine-tune checks (run before committing to § 13)

Fine-tuning cannot teach a model something the training data never showed it — these two checks are far cheaper to run here, against the base-model baseline (§ 10) and the parsed corpus (§ 2), than to discover the same gap after a multi-hour fine-tune. Both are **iterate-until-it-stops-paying-off**, not one-shot: run, fix what's genuinely wrong, re-run § 2 → § 6 for anything fixed at source, run again.

### § 12.1 — Data-coverage gap analysis

[analyze_gaps.py](pipeline/eval/analyze_gaps.py) groups § 10's baseline eval output by section/category, ranks by mean Composite, and prints the worst groups with their dominant failing metric + sample rows — enough for a human to classify each as (a) missing source content, (b) an under-represented demonstration style, or (c) a training/behavioural issue more data won't fix. Domain-agnostic (`--group-by <field>`), reused unmodified by the OOW notebook.


In [30]:
# § 13.6 code: analyze both tracks' base-Qwen eval output for data-coverage gaps.
# Safe to re-run as many times as useful (e.g. after adding a new source document
# and rebuilding § 8-§ 12) -- it only reads existing eval files, never writes to them.
import subprocess, sys
from pathlib import Path

W = Path.cwd()
CACHE = W / "Data" / "VHF" / "VHF_Agents_Training"

track1_eval = CACHE / "eval_qwen_base.jsonl"
track2_eval = CACHE / "eval_qwen_base_colreg.jsonl"


def run_gap_analysis(eval_file: Path, group_by: str) -> None:
    result = subprocess.run(
        [sys.executable, "-X", "utf8", "-m", "pipeline.eval.analyze_gaps",
         "--eval-file", str(eval_file), "--group-by", group_by,
         "--top-n-groups", "8", "--samples-per-group", "2"],
        cwd=W, capture_output=True, text=True, encoding="utf-8",
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise subprocess.CalledProcessError(result.returncode, result.args)


if track1_eval.exists():
    print("=" * 100)
    print("TRACK 1 (rules & knowledge) -- grouped by section_id")
    print("=" * 100)
    run_gap_analysis(track1_eval, "section_id")
else:
    print(f"Skipping Track 1 -- {track1_eval.name} not found yet.")
    print("Run § 13's ablation (or eval_finetuned.py --model Qwen/Qwen2.5-7B-Instruct --tag qwen_base "
          "on the cloud pod, then copy the file down) first.")

if track2_eval.exists():
    print("\n" + "=" * 100)
    print("TRACK 2 (conversational compliance) -- grouped by category")
    print("=" * 100)
    run_gap_analysis(track2_eval, "category")
else:
    print(f"Skipping Track 2 -- {track2_eval.name} not found yet (produced by § 13.5 above).")


TRACK 1 (rules & knowledge) -- grouped by section_id
Loaded 540 rows from c:\Users\jcsch\Documents\Python\Auto Pilot\Data\VHF\VHF_Agents_Training\eval_qwen_base.jsonl
DATA-COVERAGE GAP ANALYSIS  —  grouped by 'section_id'  (540 rows across 27 groups)

Group                                            n  MeanComposite
2.11                                            20          0.240
2.7                                             20          0.358
2.10                                            20          0.359
2.5                                             20          0.377
1.6                                             20          0.396
2.2                                             20          0.409
2.6                                             20          0.410
2.12                                            20          0.419
2.3                                             20          0.428
2.1                                             20          0.440
2.4                   

### § 12.2 — Cross-source consistency check

[check_consistency.py](pipeline/eval/check_consistency.py) deliberately checks only a narrow list of facts with exactly **one** universally-correct answer (NATO/ITU phonetic alphabet, MAYDAY/PAN PAN/SECURITE repeat counts, standardized VHF digit pronunciation) — high precision over recall, since e.g. two sources legitimately citing different regional working channels is correct variation, not a bug.

**What it found on this corpus** (three sweeps, summarized — details in git history): 101 wrong phonetic-alphabet spellings (87 in one self-generated document, fixed at source), 64 non-standard digit pronunciations in two more self-generated documents (fixed at source), and — via manual review beyond the fixed checklist — a MAYDAY-repeat-interval figure that had conflated two different waiting periods, and three "OVER AND OUT" usages contradicting the corpus's own rule. All fixed via one of three paths:
1. **Fix at source** (preferred for editable `.txt`/`.md`) — edit + delete the stale JSON + re-run § 2 → § 6.
2. **Normalize** — a scoped `{section_id, find, replace}` entry in `consistency_normalizations.json`, applied by § 2's cleanup pass to only that exact section (never a blind corpus-wide replace — the same word can be a genuine error in one section and a legitimate name in another, e.g. "Search Pattern Alpha").
3. **Exclude the section** — add its `section_id` to `consistency_exclusions.json` when the claim can't be cleanly separated from otherwise-useful text.


In [44]:
# § 13.7 code: Level-1 cross-source consistency check over every parsed section
# in _json/. Safe to re-run as many times as useful -- it only reads JSON and
# writes its findings report, never edits a source document or a JSON itself.
result = subprocess.run(
    [sys.executable, "-X", "utf8", "-m", "pipeline.eval.check_consistency"],
    cwd=W, capture_output=True, text=True, encoding="utf-8",
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise subprocess.CalledProcessError(result.returncode, result.args)


CROSS-SOURCE CONSISTENCY CHECK (Level 1)  —  4 finding(s)

--- phonetic_variant  ·  VHFPro.txt  ·  section s_bb68d4f9 (MV Ocean Star: "Port Control, QUESTION. Can I arri) ---
  ITU/NATO standard spelling is "ALFA", not "Alpha"
  snippet: ...Seven-Alpha 15 minutes later at One-Four-Zero-Zero local time?  My posi...

--- phonetic_variant  ·  VHFPro.txt  ·  section s_5698a335 (Port Control: "MV Ocean Star, ANSWER. Affirmative,) ---
  ITU/NATO standard spelling is "ALFA", not "Alpha"
  snippet: ...Seven-Alpha at One-Four-Zero-Zero local time. ADVICE. Reduce  speed to ...

--- phonetic_variant  ·  VHFPro.txt  ·  section s_89a08503 (Search and Rescue Coordination) ---
  ITU/NATO standard spelling is "ALFA", not "Alpha"
  snippet: ... as On-Scene Coordinator for SAR operation.  Search pattern Alpha commences at position Three-Six  degrees Four-Five minutes ...

--- phonetic_variant  ·  VHFPro.txt  ·  section s_348c401f (Dover VTS: "MV Nordic Star, this is Dover VTS. Go ) ---
  ITU/NATO standa

---

# § 13 · Fine-tuning VHF-QWEN (3 stages)

QLoRA on Qwen2.5-7B-Instruct in 4-bit NF4. Fits in 8 GB VRAM with `batch=1`, `grad_accum=16`, gradient checkpointing, bf16.

| Stage | Data | Rank | Epochs | LR | Adapter |
|---|---|---:|---:|---:|---|
| 1 SFT | direct+CoT+RAG+multihop+PG-steps | 16 | 3 | 2e-4 | `_models/VHF/vhf_qwen_sft_lora/` |
| 2 DPO | preference pairs (incl. `swap_step_order` axis) | 8 | 1 | 5e-5 | `_models/VHF/vhf_qwen_dpo_lora/` |
| 3 Reflection | Draft/Critique/Refined | 8 | 2 | 1e-4 | `_models/VHF/vhf_qwen_reflect_lora/` |

Scripts: [train_sft.py](pipeline/train/train_sft.py), [train_dpo.py](pipeline/train/train_dpo.py), [train_reflection.py](pipeline/train/train_reflection.py).

**Run in separate terminals** (not in parallel) — each stage reloads Qwen and needs the whole GPU.


In [ ]:
# Stage 1 — SFT. ~4-6 hours on RTX 4070. Adapter ~50 MB.
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.train.train_sft"], cwd=W)

In [ ]:
# Stage 2 — DPO. ~1-2 hours. Adapter ~25 MB.
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.train.train_dpo"], cwd=W)

In [ ]:
# Stage 3 — Reflection. ~30 min. Adapter ~25 MB.
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.train.train_reflection"], cwd=W)

---

## § 13.1 · Merge → VHF-QWEN (single deployable model)

All three LoRA adapters are merged into the base weights → one self-contained HF model in `_models/VHF/VHF-QWEN/`.

Code: [merge_adapter.py](pipeline/train/merge_adapter.py). Merging temporarily dequantizes to fp16 (~14 GB) — slow on a laptop GPU. For production: run the merge on a larger GPU.


In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.train.merge_adapter"], cwd=W)

---

# § 13.2 · Track 1 — Evaluate VHF-QWEN on 540 held-out gold questions (rules & knowledge)

**Without** prompt tricks (no RAG, no CoT instruction) — this measures what the model actually internalized during fine-tuning. Metrics: the closed-book variant of suite v2 (§ 9) — AnswerCorrectness, CorpusGrounded, AnswerRelevancy, NumericF1, LitHit, Cover, fixed-weight Composite.

Code: [eval_finetuned.py](pipeline/eval/eval_finetuned.py) (requires § 7's `vhf_gold_answers_claims.json`). VRAM-safe: first Qwen, then unload Qwen, then embedder + judge.

See § 13.3 for the companion **Track 2** evaluation (conversational compliance).


In [ ]:
# Full 540. For a smoke test, add --n 50.
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.eval_finetuned",
                       "--model", "_models/VHF/VHF-QWEN", "--tag", "vhf_qwen"], cwd=W)

---

## § 13.3 · Track 2 — Evaluate conversational compliance on 498 COLREG scenarios

Where § 13.2 asks *"does the model know the rule?"*, this asks *"can the model actually run the radio conversation correctly?"* The **behavioral graders are kept unchanged** (RAGAS has no notion of procedure compliance — they are the point of this track), with the claim-level metrics from § 9 added around them:

| Metric | What it checks |
|---|---|
| **ChannelProc** | rule-based: hailed on 16 AND named a distinct working channel? |
| **CallFormatOK** | rule-based: called the OTHER vessel first (not the self-hailing bug)? |
| **ColregCorrect** | judge: does the stated action comply with the cited COLREG rule(s)? |
| AnswerCorrectness / CorpusGrounded / AnswerRelevancy / NumericF1 / LitHit / Cover | same definitions as § 9 |

Fixed Track 2 composite: ColregCorrect .30 · AnswerCorrectness .25 · ChannelProc .15 · CallFormatOK .10 · NumericF1 .10 · CorpusGrounded .10.

Code: [eval_colreg_scenarios.py](pipeline/eval/eval_colreg_scenarios.py) (requires § 7's `vhf_colreg_scenarios_claims.json`).

Note: base Qwen's Track 2 number was already computed in § 10; this section starts from the fine-tuned `vhf_qwen` tag for direct comparison.


In [ ]:
# Full 498. For a smoke test, add --n 50.
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.eval_colreg_scenarios",
                       "--model", "_models/VHF/VHF-QWEN", "--tag", "vhf_qwen"], cwd=W)

---

## § 13.4 · Stage-attribution probes — did DPO and Reflection actually do their jobs?

Aggregate eval deltas can't say WHICH training stage helped. [probe_dpo.py](pipeline/eval/probe_dpo.py) measures each stage's job directly, on a deterministic **held-out** probe set (180 items = 6 axes × 30, cached in `probe_set.json`):

- **DPO probe**: per-axis (wrong_channel / wrong_proword / missing_step / dropped_regulation / dropped_warning / swapped_step_order), how often the checkpoint assigns higher log-likelihood to the good answer than the perturbed one. DPO should raise these win-rates over SFT-only.
- **Reflection probe**: feeds a flawed draft, asks for critique + revision; metric = Δ AnswerCorrectness (revision − draft). Reflection should show a larger positive Δ than base/SFT — if not, the stage is dead weight.

Run once per checkpoint — GPU-bound, cloud pod or `--force-4bit` locally. Decision rule: each stage must beat its predecessor AND the best ablation config (§ 11), with CI, to justify its GPU cost.


In [ ]:
# § 16.7 code: stage probes per checkpoint. Each run loads the model once and
# writes probe_{tag}.json. Reflection probe needs § 12.9's gold_claims.
for model_path, tag in [
    ("Qwen/Qwen2.5-7B-Instruct", "qwen_base"),
    ("_models/VHF/VHF-QWEN",     "vhf_qwen"),
    # intermediate checkpoints, when present (cloud stage-ladder runs):
    # ("_models/VHF/VHF-QWEN-sft-only", "sft_only"),
    # ("_models/VHF/VHF-QWEN-sft-dpo",  "sft_dpo"),
]:
    out = CACHE / f"probe_{tag}.json"
    if out.exists():
        print(f"Skipping {tag} -- {out.name} already exists.")
        continue
    subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.probe_dpo",
                           "--model", model_path, "--tag", tag, "--force-4bit"], cwd=W)


---

# § 14 · Compression 1 — AWQ int4 quantization

Activation-aware Weight Quantization: uses activation statistics from VHF-corpus samples to identify which weights are "important" and protects them from quantization noise.

- Input: VHF-QWEN (~14 GB bf16)
- Output: VHF-QWEN-awq-int4 (~4 GB, ~1.5-2× faster on consumer GPUs)
- Calibration: 128 samples from `vhf_sft_rag.jsonl`

One-time install: `pip install autoawq`. Code: [compress_quantize_awq.py](pipeline/compress/compress_quantize_awq.py).


In [ ]:
# One-time install required: pip install autoawq
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.compress.compress_quantize_awq"], cwd=W)

---

## § 14.1 · Compression 2 — Layer pruning (ShortGPT-style)

Measures **Block Influence** for each transformer block: `1 - cos(input, output)`. Blocks with low BI barely transform their input and can be dropped.

- Qwen2.5-7B has 28 blocks → we drop 4 → ~15-20% inference speedup
- Some quality is lost but recovered in § 14.2 (distillation)

Code: [compress_prune.py](pipeline/compress/compress_prune.py). Use `--measure-only` to just print the BI report.


In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.compress.compress_prune",
                       "--n-prune", "4"], cwd=W)

---

## § 14.2 · Compression 3 — Knowledge Distillation → DistillVHF-QWEN

VHF-QWEN (teacher, 7B, 4-bit) → Qwen2.5-1.5B (student, bf16 + LoRA).

**Loss**: `α · CE_hard + (1-α) · T² · KL(softmax(t/T) ‖ softmax(s/T))` with T=2.0, α=0.5. Same tokenizer family (Qwen2.5) → logits comparable at the token level.

VRAM budget: teacher 4-bit ~5 GB + student bf16 ~3 GB + activations ≈ **fits in 8 GB**.

Code: [compress_distill.py](pipeline/compress/compress_distill.py). Alternative student: `--student-id _models/VHF/VHF-QWEN-pruned` to recover the pruned version.


In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.compress.compress_distill",
                       "--merge-final"], cwd=W)

---

# § 15 · Evaluate DistillVHF-QWEN (both tracks) + final comparison

Run **both** § 13.2's `eval_finetuned.py` (Track 1: rules) and § 13.3's `eval_colreg_scenarios.py` (Track 2: conversational compliance) against the distilled student. Then compare every model — base Qwen through the distilled VHF fine-tune — on both tracks side by side.

**Metric-suite hygiene**: every summary JSON carries `metric_suite` + `judge_model` (§ 9). Only compare summaries with the same stamp. Existing answer files can be re-scored under the new suite without re-running inference: `python -m pipeline.eval.ragas_metrics --rescore <eval_answers.jsonl>`.


In [ ]:
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.eval_finetuned",
                       "--model", "_models/VHF/DistillVHF-QWEN", "--tag", "distill_vhf"], cwd=W)
subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.eval_colreg_scenarios",
                       "--model", "_models/VHF/DistillVHF-QWEN", "--tag", "distill_vhf"], cwd=W)

### § 15.1 — Backfill Track 2 for the other model tags

§ 13's eval only ran Track 2 for `vhf_qwen`. Run it for the other tags too (`qwen_base`, `vhf_qwen_awq`) so the final comparison below is complete — same pattern, just swap `--model`/`--tag`.

`qwen_base` was already computed in § 10, so this loop detects `eval_qwen_base_colreg_summary.json` and skips straight to `vhf_qwen_awq` — nothing is run twice.


In [ ]:
for model_path, tag in [
    ("Qwen/Qwen2.5-7B-Instruct", "qwen_base"),
    ("_models/VHF/VHF-QWEN-awq-int4", "vhf_qwen_awq"),
]:
    out = CACHE / f"eval_{tag}_colreg_summary.json"
    if out.exists():
        print(f"Skipping {tag} -- {out.name} already exists.")
        continue
    subprocess.check_call([sys.executable, "-X", "utf8", "-m", "pipeline.eval.eval_colreg_scenarios",
                           "--model", model_path, "--tag", tag], cwd=W)

In [ ]:
# Final comparison across all runs -- BOTH evaluation tracks, plus latency.
import pandas as pd
import matplotlib.pyplot as plt

rows = []

# ── Prompt-injection ablation (§ 11): base / RAG / CoT / RAG+CoT / PG on base Qwen ──
# Track 1 only -- the V0..V4 prompt configs were never re-run against the COLREG
# scenarios. Track 2's own base-Qwen baseline is captured separately below via
# the "qwen_base" tag, which § 10 populates before any fine-tuning happens.
ablation_summary_file = CACHE / "ablation_summary.json"
if ablation_summary_file.exists():
    ablation_summary = json.loads(ablation_summary_file.read_text())
    for cfg, s in ablation_summary.items():
        if cfg == "_meta":  # metric_suite/judge_model stamp, not a model row
            continue
        rows.append({
            "model": f"ablation_{cfg}",
            "Track1_RulesKnowledge": s.get("Composite"),
            "Track2_ConversationCompliance": None,
            "latency_mean_s": s.get("latency_mean_s"),
        })

# ── Fine-tuned / compressed models — both tracks, side by side ──────────
for tag in ["qwen_base", "vhf_qwen", "vhf_qwen_awq", "distill_vhf"]:
    p1 = CACHE / f"eval_{tag}_summary.json"
    p2 = CACHE / f"eval_{tag}_colreg_summary.json"
    if not p1.exists() and not p2.exists():
        continue
    row = {"model": tag, "Track1_RulesKnowledge": None, "Track2_ConversationCompliance": None,
           "latency_mean_s": None}
    if p1.exists():
        s1 = json.loads(p1.read_text())
        row["Track1_RulesKnowledge"] = s1["means"].get("Composite")
        row["latency_mean_s"] = s1.get("latency", {}).get("mean_s")
    if p2.exists():
        s2 = json.loads(p2.read_text())
        row["Track2_ConversationCompliance"] = s2["means"].get("Composite")
        if row["latency_mean_s"] is None:
            row["latency_mean_s"] = s2.get("latency", {}).get("mean_s")
    rows.append(row)

if rows:
    df = pd.DataFrame(rows).set_index("model")
    display(df)

    fig, axes = plt.subplots(1, 3, figsize=(19, 4))

    df["Track1_RulesKnowledge"].plot(kind="barh", ax=axes[0], color="#4e9af1",
                                      title="Track 1 -- Rules knowledge\n(540 exam Q's, higher = better)")
    axes[0].set_xlim(0, 1)
    axes[0].grid(True, axis="x", alpha=0.3)

    df["Track2_ConversationCompliance"].dropna().plot(
        kind="barh", ax=axes[1], color="#8e44ad",
        title="Track 2 -- Conversational compliance\n(498 COLREG scenarios, higher = better)")
    axes[1].set_xlim(0, 1)
    axes[1].grid(True, axis="x", alpha=0.3)

    if "latency_mean_s" in df.columns:
        df["latency_mean_s"].plot(kind="barh", ax=axes[2], color="#f1914e",
                                   title="Mean latency per answer\n(lower = better)")
        axes[2].set_xlabel("seconds")
        axes[2].grid(True, axis="x", alpha=0.3)

    fig.tight_layout()
    plt.savefig(CACHE / "vhf_final_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved to {CACHE / 'vhf_final_comparison.png'}")

else:
    print("No eval/ablation summaries found yet. Run § 11, § 13 first.")
